In [ ]:
"""
Task 1: Predict Restaurant Ratings
Builds regression models to predict `Aggregate rating` and reports metrics
plus the most influential features.
"""

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

DATA_PATH = Path(r"D:\Code\Repo\Machine-Learning-Internship\Dataset .csv")
RANDOM_STATE = 42


In [ ]:
def load_and_clean(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, encoding="utf-8-sig")
    # The target: rows with 0 rating almost always mean "not rated" — keep them,
    # they are legitimate low-signal samples. Only drop if target is NaN.
    df = df.dropna(subset=["Aggregate rating"]).copy()

    # Drop identifiers / free-text fields that add noise, plus leakage columns
    # (Rating color & Rating text are derived directly from Aggregate rating).
    drop_cols = [
        "Restaurant ID",
        "Restaurant Name",
        "Address",
        "Locality Verbose",
        "Rating color",
        "Rating text",
    ]
    df = df.drop(columns=[c for c in drop_cols if c in df.columns])

    # Handle missing values
    if "Cuisines" in df.columns:
        df["Cuisines"] = df["Cuisines"].fillna("Unknown")
    # Numeric NaNs -> median
    for col in df.select_dtypes(include=[np.number]).columns:
        if df[col].isna().any():
            df[col] = df[col].fillna(df[col].median())
    # Remaining object NaNs -> "Unknown"
    for col in df.select_dtypes(include=["object"]).columns:
        if df[col].isna().any():
            df[col] = df[col].fillna("Unknown")
    return df


In [ ]:
def encode_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    # High-cardinality categoricals → label encode to keep the feature matrix small.
    # For linear regression this is imperfect, but tree models handle it fine.
    for col in df.select_dtypes(include=["object"]).columns:
        df[col] = LabelEncoder().fit_transform(df[col].astype(str))
    return df


In [ ]:
def evaluate(name, model, X_test, y_test):
    pred = model.predict(X_test)
    mse = mean_squared_error(y_test, pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, pred)
    r2 = r2_score(y_test, pred)
    print(f"  {name:22s}  MSE={mse:.4f}  RMSE={rmse:.4f}  MAE={mae:.4f}  R²={r2:.4f}")
    return {"model": name, "MSE": mse, "RMSE": rmse, "MAE": mae, "R2": r2}


In [ ]:
def main():
    print(f"Loading: {DATA_PATH}")
    df = load_and_clean(DATA_PATH)
    print(f"Shape after cleaning: {df.shape}")
    print(f"Target stats:\n{df['Aggregate rating'].describe()}\n")

    df_enc = encode_features(df)
    X = df_enc.drop(columns=["Aggregate rating"])
    y = df_enc["Aggregate rating"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE
    )
    print(f"Train: {X_train.shape}  Test: {X_test.shape}\n")

    models = {
        "LinearRegression": LinearRegression(),
        "DecisionTree": DecisionTreeRegressor(random_state=RANDOM_STATE),
        "RandomForest": RandomForestRegressor(
            n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1
        ),
    }

    print("Test-set performance:")
    results = []
    fitted = {}
    for name, m in models.items():
        m.fit(X_train, y_train)
        results.append(evaluate(name, m, X_test, y_test))
        fitted[name] = m

    # Pick the best model by R² for interpretation
    best_name = max(results, key=lambda r: r["R2"])["model"]
    best = fitted[best_name]
    print(f"\nBest model by R²: {best_name}")

    # Feature importances / coefficients
    print("\nTop 10 most influential features:")
    if hasattr(best, "feature_importances_"):
        imp = pd.Series(best.feature_importances_, index=X.columns).sort_values(
            ascending=False
        )
        print(imp.head(10).to_string())
    elif hasattr(best, "coef_"):
        imp = pd.Series(np.abs(best.coef_), index=X.columns).sort_values(
            ascending=False
        )
        print(imp.head(10).to_string())

    # Also show linear-regression coefficients (signed) for directional insight
    lr = fitted["LinearRegression"]
    coefs = pd.Series(lr.coef_, index=X.columns).sort_values(key=np.abs, ascending=False)
    print("\nLinear regression — top 10 signed coefficients (direction matters):")
    print(coefs.head(10).to_string())


In [ ]:
if __name__ == "__main__":
    main()
